# KV-cache eviction headroom (CPU only)

**Question:** before building predictive eviction into a real engine, how much
better than LRU could *any* eviction policy do on our workloads, and how much of
that does a realistic predictor capture?

This notebook replays each workload through `src/kvcache/cachesim.py`, a
block-level model of vLLM's prefix cache. It needs no GPU and no vLLM; the full
sweep takes about 3 minutes on a laptop or a Kaggle CPU session.

**How the model works**
- Prompts are split into 16-token blocks whose hashes chain from token 0, as in
  vLLM. A block is reused only if everything before it matched too.
- A request reuses its longest resident prefix. Afterwards all of its blocks are
  resident and most recently used.
- The KV budget is a fixed number of blocks. For Qwen2.5-1.5B at fp16, 1 GiB
  holds 2,340 blocks.

**Eviction policies compared** (these decide what the *engine* drops; the GPU
experiment only reorders requests and cannot do this)

| Policy | Evicts first | Uses the future? |
|---|---|---|
| `lru` | least recently used block (what vLLM does) | no |
| `predictive` | blocks of sessions least likely to return, from `return_likelihood` fed with **past** turns only (the GPU session-aware signal) | no |
| `perfect-return` | blocks of the session that returns latest, from **true** next-arrival times | yes: when sessions return |
| `oracle` | block whose next use is furthest away (Belady) | yes: everything |
| `infinite` (dashed line) | nothing; the ceiling set by first-time and truncation misses | n/a |

**How to read it**
- `oracle − lru` is the **headroom**: the most any eviction policy could add.
  About 0 means there is nothing to win at that budget.
- `predictive − lru` is what a history-only predictor actually adds.
- `perfect-return − lru` is the best a *return-time* predictor could ever add.
  If even this is ≤ 0 while the oracle gains, then "will the session return?" is
  the wrong target, and the predictor must forecast **reusable tokens** instead.

**Caveats**
- Requests are served one at a time in arrival order. The GPU overlaps a few, so
  the absolute rates differ slightly; the comparison between policies is what
  counts.
- The oracle is Belady on blocks. It is a strong upper bound, not a proven optimum
  once blocks depend on their parents.
- The T4 capacity marker is inferred from round 3's measured hit rate (cell 4).
  It is an estimate.

In [ ]:
# Cell 1: find the repo (running inside it locally), or clone it (Kaggle CPU).
import os, sys, subprocess

def _find_repo():
    d = os.getcwd()
    for _ in range(4):
        if os.path.isdir(os.path.join(d, 'src', 'kvcache')):
            return d
        d = os.path.dirname(d)
    return None

REPO = _find_repo()
if REPO is None:
    REPO = ('/kaggle/working/sharing-aware-kv-cache' if os.path.isdir('/kaggle')
            else os.path.abspath('sharing-aware-kv-cache'))
    if os.path.isdir(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'])
    else:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/Shofiya2003/sharing-aware-kv-cache.git', REPO],
                       check=True)
for p in (os.path.join(REPO, 'src'), os.path.join(REPO, 'experiments')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO)
print('[setup] repo:', REPO)

In [ ]:
# Cell 2: what to sweep.
import cache_headroom as H

WORKLOADS = ['arm1_round4', 'arm1_round3', 'arm2_preamble', 'arm2_mid']
SEEDS = [0, 1, 2, 3, 4]
CAPACITIES = H.DEFAULT_CAPACITIES          # KV blocks; 2,340 per GiB
OUT_CSV = 'results/cpu_headroom/headroom.csv'
for w in WORKLOADS:
    print(f'{w:14s}', H.PRESETS[w])

In [ ]:
# Cell 3: run the sweep (~3 min for everything).
import pandas as pd

rows = H.run(WORKLOADS, SEEDS, CAPACITIES, warmup_s=30.0, out_csv=OUT_CSV)
df = pd.DataFrame(rows)
fin = df[df.capacity_blocks != 'inf'].copy()
fin['capacity_blocks'] = fin.capacity_blocks.astype(int)
ceiling = df[df.capacity_blocks == 'inf'].set_index(['workload', 'seed']).cached_token_rate
print(f'{len(df)} runs')

### Where is the real T4 on the capacity axis?

Round 3 measured `cached_token_rate = 0.5457` on the T4 at `gpu_memory_utilization=0.3`
(`fifo_constrained`, seed 0, 12 sessions × 600 s). Cell 4 finds the KV budget at
which simulated LRU gives that same rate on that same workload, and uses it as the
"T4 constrained" marker. If the estimate lands between 1 and 1.7 GiB, which is
what the memory arithmetic predicts, the model is consistent with the hardware.

In [ ]:
# Cell 4: anchor the capacity axis on the round-3 T4 measurement.
import numpy as np
from kvcache.cachesim import KV_BYTES_PER_TOKEN, blocks_for_gib

ROUND3_T4 = 0.5457
ANCHOR = None
if 'arm1_round3' in WORKLOADS and 0 in SEEDS:
    c = (fin[(fin.workload == 'arm1_round3') & (fin.seed == 0) & (fin.policy == 'lru')]
         .sort_values('capacity_blocks'))
    rates, caps = c.cached_token_rate.values, c.capacity_blocks.values
    if rates.min() <= ROUND3_T4 <= rates.max():
        ANCHOR = int(round(np.exp(np.interp(ROUND3_T4, rates, np.log(caps)))))
if ANCHOR is None:
    ANCHOR = blocks_for_gib(1.3)
    print('[anchor] round-3 anchor unavailable; using a 1.3 GiB estimate')
print(f'[anchor] T4 constrained ~ {ANCHOR} blocks = '
      f'{ANCHOR * 16 * KV_BYTES_PER_TOKEN / 2**30:.2f} GiB of KV')

In [ ]:
# Cell 5: hit rate vs KV budget, one panel per workload (mean over seeds).
import matplotlib.pyplot as plt

SERIES = [  # fixed order: validated palette slots 1-4, plus a marker per policy
    ('lru', 'LRU (vLLM)', '#2a78d6', 'o'),
    ('predictive', 'Predictive (history only)', '#eb6834', 's'),
    ('perfect-return', 'Perfect return time', '#1baf7a', '^'),
    ('oracle', 'Oracle (Belady)', '#eda100', 'D'),
]
INK, MUTED, GRID = '#1f1f1e', '#6b6a64', '#e6e5df'

n = len(WORKLOADS)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(12, 4.2 * ((n + 1) // 2)),
                         sharex=True, squeeze=False)
for ax, w in zip(axes.flat, WORKLOADS):
    sub = fin[fin.workload == w].groupby(['policy', 'capacity_blocks']).cached_token_rate.mean()
    for key, label, color, marker in SERIES:
        s = sub.loc[key]
        ax.plot(s.index, s.values, color=color, lw=2, marker=marker, ms=6,
                markeredgecolor='white', markeredgewidth=1, label=label)
    ceil = ceiling.loc[w].mean()
    ax.axhline(ceil, color=MUTED, lw=1.5, ls='--', label='Infinite cache (ceiling)')
    ax.axvline(ANCHOR, color=MUTED, lw=1, ls=':')
    ax.text(ANCHOR, ax.get_ylim()[0], ' ≈ T4 constrained', color=MUTED,
            fontsize=9, va='bottom', ha='left')
    ax.set_title(w, color=INK, fontsize=12, loc='left')
    ax.set_xscale('log')
    ax.set_xticks(CAPACITIES)
    ax.set_xticklabels([f'{c:,}' for c in CAPACITIES], rotation=45)
    ax.minorticks_off()
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.set_ylabel('cached_token_rate', color=MUTED)
for ax in axes[-1]:
    ax.set_xlabel('KV budget (16-token blocks, log scale)', color=MUTED)
for ax in axes.flat[n:]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False, fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs('results/cpu_headroom', exist_ok=True)
fig.savefig('results/cpu_headroom/headroom.png', dpi=140)
plt.show()

In [ ]:
# Cell 6: the numbers at the T4 budget, paired by seed (same workload, so
# workload noise cancels in the difference).
from kvcache.cachesim import POLICIES, simulate
from kvcache.workload import WorkloadConfig, generate_workload

recs = []
for w in WORKLOADS:
    for seed in SEEDS:
        ev = generate_workload(WorkloadConfig(seed=seed, **H.PRESETS[w])).events
        r = {p: simulate(ev, ANCHOR, p, warmup_s=30.0).cached_token_rate for p in POLICIES}
        r['infinite'] = ceiling.loc[(w, seed)]
        recs.append(dict(workload=w, seed=seed, **r))
at = pd.DataFrame(recs)

table = at.groupby('workload')[['lru', 'predictive', 'perfect-return', 'oracle', 'infinite']].mean()
print(f'cached_token_rate at {ANCHOR} blocks (mean over {len(SEEDS)} seeds)')
display(table.round(3))

gaps = []
for w, g in at.groupby('workload'):
    for p in ('predictive', 'perfect-return', 'oracle', 'infinite'):
        d = g[p] - g['lru']
        gaps.append(dict(workload=w, vs_lru=p, mean=d.mean(), sd=d.std(),
                         seeds_positive=f'{(d > 0).sum()}/{len(d)}'))
print('\nPaired difference vs LRU (same seed):')
display(pd.DataFrame(gaps).set_index(['workload', 'vs_lru']).round(4))

In [ ]:
# Cell 7: what this says.
def verdict(w):
    g = at[at.workload == w]
    lru = g.lru.mean()
    head = (g.oracle - g.lru).mean()
    pred = (g.predictive - g.lru).mean()
    perf = (g['perfect-return'] - g.lru).mean()
    ceil_gap = (g.infinite - g.lru).mean()
    print(f'--- {w} (LRU {lru:.3f}) ---')
    if ceil_gap < 0.005:
        print('  No eviction pressure at this budget: LRU already reaches the infinite-cache')
        print('  ceiling, so no eviction or ordering policy can raise the hit rate here.')
        return
    print(f'  headroom (oracle - LRU)        {head:+.3f}   of {ceil_gap:+.3f} lost to eviction')
    if head < 0.01:
        print('  Headroom under 0.01: little for any eviction policy to win at this budget.')
    share = lambda x: f'{x / head:+.0%} of headroom' if head >= 0.01 else ''
    print(f'  predictive (history) - LRU     {pred:+.3f}   {share(pred)}')
    print(f'  perfect return time - LRU      {perf:+.3f}   {share(perf)}')
    if perf <= 0 and head >= 0.01:
        print('  -> Even knowing exactly when each session returns does not beat LRU, so')
        print('     "will the session return?" is not enough. The oracle wins by knowing')
        print('     which BLOCKS will be reused. (One candidate reason, not yet tested: a')
        print('     returning session whose context was truncated no longer matches its')
        print('     old blocks, so keeping them for it is wasted space.)')

for w in WORKLOADS:
    verdict(w)

if {'arm2_preamble', 'arm2_mid'} <= set(WORKLOADS):
    t = at.groupby('workload').mean(numeric_only=True)
    print('\n--- cross-session sharing (arm 2) ---')
    print(f"  doc at token 0 vs mid-prompt, infinite cache: "
          f"{t.loc['arm2_preamble', 'infinite']:.3f} vs {t.loc['arm2_mid', 'infinite']:.3f} "
          f"({t.loc['arm2_preamble', 'infinite'] - t.loc['arm2_mid', 'infinite']:+.3f})")
    print(f"  same at the T4 budget under LRU:              "
          f"{t.loc['arm2_preamble', 'lru']:.3f} vs {t.loc['arm2_mid', 'lru']:.3f} "
          f"({t.loc['arm2_preamble', 'lru'] - t.loc['arm2_mid', 'lru']:+.3f})")
    print('  The difference is the cross-session reuse vLLM can deliver at all.')